
# Climate Data Integration for Biodiversity Forecasting

This notebook extends the modeling logic from `Model_Dataset_Building&Evaluation.ipynb` with **modular climate covariates**.



## Why start with annual temperature?
A country-year annual temperature series is the cleanest first integration because the biodiversity benchmark already operates at a yearly resolution and includes a `country` field for many records. That makes the first merge interpretable and easy to audit.

## Expected workflow
1. Load one cleaned biodiversity dataset  
2. Convert wide yearly columns to long format  
3. Build population lag/trend features  
4. Load annual temperature data  
5. Merge temperature by `country` + `Year`  
6. Create climate lags/rolling features  
7. Train the same model family as the benchmark  
8. Check whether climate features improve error and/or appear important


In [ ]:

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


In [ ]:

# =========================
# Configuration
# =========================

DATA_DIR = Path("../data/interim")
DATASET_FILES = sorted(DATA_DIR.rglob("lpd_*.csv"))
DATASET_FILES = [p for p in DATASET_FILES if p.name != "lpd_master_initial.csv"]

# Pick one dataset to test first.
# Update this if your team has already agreed on a final candidate dataset.
TARGET_DATASET = None  # example: DATA_DIR / "strict" / "lpd_terrestrial_strict_last2020_unitsusable_global_zeroskeep.csv"

N_LAGS = 4
TEST_YEARS = 5
RANDOM_STATE = 42

# Annual temperature by country
TEMPERATURE_URL = "https://ourworldindata.org/grapher/average-annual-surface-temperature.csv"

# Optional local cache path
CACHE_DIR = Path("../data/external/climate")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
TEMPERATURE_CACHE_PATH = CACHE_DIR / "average_annual_surface_temperature.csv"

# Country name harmonization
COUNTRY_NAME_FIXES = {
    "United States": "United States",
    "United Kingdom": "United Kingdom",
    "Russian Federation": "Russia",
    "Viet Nam": "Vietnam",
    "Czech Republic": "Czechia",
    "Iran (Islamic Republic of)": "Iran",
    "Bolivia (Plurinational State of)": "Bolivia",
    "Venezuela (Bolivarian Republic of)": "Venezuela",
    "Republic of Korea": "South Korea",
    "Democratic Republic of the Congo": "Democratic Republic of Congo",
    "Congo, Dem. Rep.": "Democratic Republic of Congo",
    "Congo, Rep.": "Congo",
    "Lao People's Democratic Republic": "Laos",
    "Syrian Arab Republic": "Syria",
    "United Republic of Tanzania": "Tanzania",
    "Moldova, Republic of": "Moldova",
}

print(f"Found {len(DATASET_FILES)} candidate datasets.")
for p in DATASET_FILES[:10]:
    print("-", p.relative_to(DATA_DIR))
if len(DATASET_FILES) > 10:
    print("...")


In [ ]:

if TARGET_DATASET is None:
    if not DATASET_FILES:
        raise FileNotFoundError("No cleaned biodiversity datasets found under ../data/interim/")
    TARGET_DATASET = DATASET_FILES[0]

print("Using dataset:", TARGET_DATASET)


## 1. Reuse the same biodiversity preprocessing logic

In [ ]:

df_reference = pd.read_csv(TARGET_DATASET, nrows=5)
reference_columns = list(df_reference.columns)
reference_columns_set = set(reference_columns)

year_columns = [col for col in reference_columns if str(col).isdigit()]

static_numeric_features = ["latitude", "longitude"]
static_categorical_features = [
    "class",
    "family",
    "ipbes_subregion",
    "system_group",
    "t_realm",
    "t_biome",
    "units",
]
identifier_columns = ["id", "binomial", "common_name", "location", "country"]

static_numeric_features = [c for c in static_numeric_features if c in reference_columns_set]
static_categorical_features = [c for c in static_categorical_features if c in reference_columns_set]
identifier_columns = [c for c in identifier_columns if c in reference_columns_set]

print("Year columns:", len(year_columns))
print("Static numeric features:", static_numeric_features)
print("Static categorical features:", static_categorical_features)
print("Identifier columns:", identifier_columns)


In [ ]:

def build_series_id(df):
    parts = []

    if "common_name" in df.columns:
        parts.append(df["common_name"].astype(str))
    elif "binomial" in df.columns:
        parts.append(df["binomial"].astype(str))

    if "country" in df.columns:
        parts.append(df["country"].astype(str))

    if len(parts) == 0:
        return pd.Series(np.arange(len(df)).astype(str), index=df.index)

    series_id = parts[0]
    for p in parts[1:]:
        series_id = series_id + "|" + p

    return series_id


def harmonize_country_names(series, name_fixes=None):
    if name_fixes is None:
        name_fixes = {}
    out = series.astype(str).str.strip()
    return out.replace(name_fixes)


def wide_to_long(df):
    id_columns = [col for col in df.columns if col not in year_columns]

    long_df = (
        df.melt(
            id_vars=id_columns,
            value_vars=year_columns,
            var_name="Year",
            value_name="Population",
        )
        .dropna(subset=["Population"])
        .copy()
    )

    long_df["Year"] = long_df["Year"].astype(int)
    long_df["series_id"] = build_series_id(long_df)
    long_df["log_population"] = np.log1p(long_df["Population"])

    if "country" in long_df.columns:
        long_df["country_clean"] = harmonize_country_names(long_df["country"], COUNTRY_NAME_FIXES)

    return long_df


def add_population_lag_features(long_df, n_lags=4):
    long_df = long_df.sort_values(["series_id", "Year"]).copy()
    grouped_pop = long_df.groupby("series_id")["Population"]

    for lag in range(1, n_lags + 1):
        long_df[f"lag_{lag}"] = grouped_pop.shift(lag)

    long_df["prev_year"] = long_df.groupby("series_id")["Year"].shift(1)
    long_df["year_gap_from_prev"] = long_df["Year"] - long_df["prev_year"]

    long_df["rolling_mean_3"] = (
        grouped_pop.shift(1).rolling(3).mean().reset_index(level=0, drop=True)
    )
    long_df["rolling_std_3"] = (
        grouped_pop.shift(1).rolling(3).std().reset_index(level=0, drop=True)
    )

    long_df["population_difference"] = long_df["lag_1"] - long_df["lag_2"]
    long_df["population_growth_rate"] = (
        (long_df["lag_1"] - long_df["lag_2"]) / long_df["lag_2"].replace(0, np.nan)
    )
    long_df["population_growth_rate"] = long_df["population_growth_rate"].replace([np.inf, -np.inf], np.nan)

    needed = [f"lag_{lag}" for lag in range(1, n_lags + 1)]
    long_df = long_df.dropna(subset=needed).copy()

    return long_df


def temporal_split(df, test_years=5):
    years = sorted(df["Year"].unique())
    test_year_values = years[-test_years:]
    train_df = df[df["Year"] < min(test_year_values)].copy().reset_index(drop=True)
    test_df = df[df["Year"].isin(test_year_values)].copy().reset_index(drop=True)
    return train_df, test_df, test_year_values


def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    mean_abs_true = np.mean(np.abs(y_true))
    nmae = mae / mean_abs_true if mean_abs_true != 0 else np.nan

    denom = np.sum(np.abs(y_true))
    wape = np.sum(np.abs(y_true - y_pred)) / denom if denom != 0 else np.nan

    smape_denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    valid = smape_denom != 0
    smape = np.mean(np.abs(y_true[valid] - y_pred[valid]) / smape_denom[valid]) if np.any(valid) else np.nan

    return {
        "MAE": mae,
        "RMSE": rmse,
        "NMAE": nmae,
        "WAPE": wape,
        "sMAPE": smape,
    }


## 2. Load and inspect the biodiversity dataset

In [ ]:

df_bio = pd.read_csv(TARGET_DATASET)
long_df = wide_to_long(df_bio)
long_df = add_population_lag_features(long_df, n_lags=N_LAGS)

print("Wide shape:", df_bio.shape)
print("Long shape after lag creation:", long_df.shape)
display(long_df.head())


In [ ]:

summary = {
    "dataset": TARGET_DATASET.name,
    "rows_wide": len(df_bio),
    "rows_long": len(long_df),
    "unique_series": long_df["series_id"].nunique(),
    "year_min": int(long_df["Year"].min()),
    "year_max": int(long_df["Year"].max()),
    "countries_in_bio": int(long_df["country_clean"].nunique()) if "country_clean" in long_df.columns else np.nan,
}
pd.DataFrame([summary])


## 3. Download and prepare annual temperature data

In [ ]:

def load_temperature_data(url=TEMPERATURE_URL, cache_path=TEMPERATURE_CACHE_PATH, force_download=False):
    if cache_path.exists() and not force_download:
        temp_df = pd.read_csv(cache_path)
    else:
        temp_df = pd.read_csv(url)
        temp_df.to_csv(cache_path, index=False)

    return temp_df


def prepare_temperature_data(temp_raw):
    df = temp_raw.copy()

    expected_cols = {"Entity", "Year"}
    if not expected_cols.issubset(df.columns):
        raise ValueError(
            f"Temperature file is missing expected columns. Found columns: {list(df.columns)}"
        )

    value_candidates = [c for c in df.columns if c not in {"Entity", "Code", "Year"}]
    if len(value_candidates) != 1:
        raise ValueError(
            "Expected exactly one temperature value column after removing Entity/Code/Year. "
            f"Found: {value_candidates}"
        )

    value_col = value_candidates[0]

    df = df.rename(
        columns={
            "Entity": "country_clean",
            value_col: "temp_avg_c",
        }
    )

    df["country_clean"] = harmonize_country_names(df["country_clean"], COUNTRY_NAME_FIXES)
    df["Year"] = df["Year"].astype(int)
    df["temp_avg_c"] = pd.to_numeric(df["temp_avg_c"], errors="coerce")

    df = df.dropna(subset=["country_clean", "Year", "temp_avg_c"]).copy()

    # Remove OWID aggregates unless you explicitly want them
    drop_entities = {
        "World",
        "Asia",
        "Africa",
        "Europe",
        "North America",
        "South America",
        "Oceania",
        "High-income countries",
        "Upper-middle-income countries",
        "Lower-middle-income countries",
        "Low-income countries",
        "European Union (27)",
    }
    df = df[~df["country_clean"].isin(drop_entities)].copy()

    return df[["country_clean", "Year", "temp_avg_c"]].drop_duplicates()


In [ ]:

temp_raw = load_temperature_data()
temp_df = prepare_temperature_data(temp_raw)

print("Temperature rows:", len(temp_df))
print("Temperature countries:", temp_df["country_clean"].nunique())
print("Temperature years:", temp_df["Year"].min(), "-", temp_df["Year"].max())
display(temp_df.head())


## 4. Merge climate data into the modeling table

In [ ]:

def add_temperature_features(merged_df):
    df = merged_df.sort_values(["country_clean", "Year"]).copy()

    group = df.groupby("country_clean")["temp_avg_c"]

    df["temp_lag_1"] = group.shift(1)
    df["temp_lag_2"] = group.shift(2)
    df["temp_change_1y"] = df["temp_avg_c"] - df["temp_lag_1"]
    df["temp_rolling_mean_3"] = (
        group.shift(1).rolling(3).mean().reset_index(level=0, drop=True)
    )
    df["temp_rolling_std_3"] = (
        group.shift(1).rolling(3).std().reset_index(level=0, drop=True)
    )

    country_mean = df.groupby("country_clean")["temp_avg_c"].transform("mean")
    df["temp_country_mean"] = country_mean
    df["temp_anomaly_vs_country_mean"] = df["temp_avg_c"] - df["temp_country_mean"]

    return df


def merge_temperature(long_df, temp_df):
    if "country_clean" not in long_df.columns:
        raise ValueError(
            "The biodiversity dataset has no usable country column, so a country-year climate merge cannot be performed."
        )

    merged = long_df.merge(
        temp_df,
        on=["country_clean", "Year"],
        how="left",
        validate="m:1",
    )

    merged = add_temperature_features(merged)
    return merged


In [ ]:

model_df = merge_temperature(long_df, temp_df)

coverage = pd.DataFrame({
    "rows_total": [len(model_df)],
    "rows_with_temperature": [model_df["temp_avg_c"].notna().sum()],
    "temperature_coverage_share": [model_df["temp_avg_c"].notna().mean()],
    "countries_in_model": [model_df["country_clean"].nunique()],
    "countries_with_temperature": [model_df.loc[model_df["temp_avg_c"].notna(), "country_clean"].nunique()],
})

display(coverage)
display(model_df.head())


In [ ]:

missing_country_matches = (
    model_df.loc[model_df["temp_avg_c"].isna(), ["country_clean"]]
    .drop_duplicates()
    .sort_values("country_clean")
    .reset_index(drop=True)
)

print("Unmatched biodiversity countries:", len(missing_country_matches))
display(missing_country_matches.head(50))



### Merge sanity check
If the unmatched-country table is not empty, that does **not** automatically mean the merge is wrong. It may happen because:
- country names differ between sources
- some biodiversity rows have missing country values
- some entities in the climate file are not country-level matches
- a dataset may contain historical years outside the climate coverage you are using

Fix the obvious naming mismatches first, then rerun.


## 5. Build modular feature sets

In [ ]:

BASE_NUMERIC_FEATURES = (
    ["Year"]
    + [f"lag_{i}" for i in range(1, N_LAGS + 1)]
    + [
        "year_gap_from_prev",
        "rolling_mean_3",
        "rolling_std_3",
        "population_difference",
        "population_growth_rate",
    ]
    + static_numeric_features
)

BASE_CATEGORICAL_FEATURES = list(static_categorical_features)

CLIMATE_NUMERIC_FEATURES = [
    "temp_avg_c",
    "temp_lag_1",
    "temp_lag_2",
    "temp_change_1y",
    "temp_rolling_mean_3",
    "temp_rolling_std_3",
    "temp_country_mean",
    "temp_anomaly_vs_country_mean",
]

BASE_NUMERIC_FEATURES = [c for c in BASE_NUMERIC_FEATURES if c in model_df.columns]
BASE_CATEGORICAL_FEATURES = [c for c in BASE_CATEGORICAL_FEATURES if c in model_df.columns]
CLIMATE_NUMERIC_FEATURES = [c for c in CLIMATE_NUMERIC_FEATURES if c in model_df.columns]

FEATURE_SETS = {
    "baseline": {
        "numeric": BASE_NUMERIC_FEATURES,
        "categorical": BASE_CATEGORICAL_FEATURES,
    },
    "baseline_plus_temperature": {
        "numeric": BASE_NUMERIC_FEATURES + CLIMATE_NUMERIC_FEATURES,
        "categorical": BASE_CATEGORICAL_FEATURES,
    },
}

FEATURE_SETS


In [ ]:

def make_preprocessor(numeric_features, categorical_features):
    return ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]),
                numeric_features,
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]),
                categorical_features,
            ),
        ]
    )


## 6. Train the benchmark model with and without climate data

In [ ]:

def get_feature_columns(feature_set):
    return feature_set["numeric"] + feature_set["categorical"]


def fit_random_forest(feature_set_name, model_df):
    feature_set = FEATURE_SETS[feature_set_name]
    feature_columns = get_feature_columns(feature_set)

    required_columns = ["Population", "log_population", "Year"] + feature_columns
    working_df = model_df[required_columns].copy()

    train_df, test_df, test_years = temporal_split(working_df, test_years=TEST_YEARS)

    X_train = train_df[feature_columns]
    y_train = train_df["log_population"]
    X_test = test_df[feature_columns]
    y_test_log = test_df["log_population"]
    y_test = test_df["Population"]

    cv_years = sorted(train_df["Year"].unique())
    split_year = cv_years[int(len(cv_years) * 0.8)]

    cv_train_idx = np.where(train_df["Year"] <= split_year)[0]
    cv_val_idx = np.where(train_df["Year"] > split_year)[0]
    cv_splits = [(cv_train_idx, cv_val_idx)]

    preprocessor = make_preprocessor(
        numeric_features=feature_set["numeric"],
        categorical_features=feature_set["categorical"],
    )

    pipeline = Pipeline([
        ("prep", preprocessor),
        ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
    ])

    param_grid = {
        "model__n_estimators": [200],
        "model__max_depth": [8, None],
        "model__min_samples_leaf": [1, 2],
    }

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="neg_mean_absolute_error",
        cv=cv_splits,
        n_jobs=-1,
        refit=True,
    )
    grid.fit(X_train, y_train)

    preds_log = grid.best_estimator_.predict(X_test)
    preds = np.expm1(preds_log)
    preds = np.clip(preds, a_min=0, a_max=None)

    metrics = evaluate(y_test.to_numpy(), preds)

    result = {
        "feature_set": feature_set_name,
        "best_params": str(grid.best_params_),
        "train_rows": len(train_df),
        "test_rows": len(test_df),
        "test_years": ", ".join(map(str, test_years)),
        **metrics,
    }

    perm = permutation_importance(
        grid.best_estimator_,
        X_test,
        y_test_log,
        n_repeats=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    importance_df = pd.DataFrame({
        "feature": feature_columns,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
        "feature_set": feature_set_name,
    }).sort_values("importance_mean", ascending=False)

    pred_df = pd.DataFrame({
        "Year": test_df["Year"].values,
        "y_true": y_test.values,
        "y_pred": preds,
        "feature_set": feature_set_name,
    })

    return result, importance_df, pred_df


In [ ]:

results = []
importance_tables = []
prediction_tables = []

for fs_name in FEATURE_SETS.keys():
    result, imp_df, pred_df = fit_random_forest(fs_name, model_df)
    results.append(result)
    importance_tables.append(imp_df)
    prediction_tables.append(pred_df)

results_df = pd.DataFrame(results).sort_values("MAE").reset_index(drop=True)
importance_df = pd.concat(importance_tables, ignore_index=True)
predictions_df = pd.concat(prediction_tables, ignore_index=True)

display(results_df)


## 7. Did temperature help?

In [ ]:

baseline_metrics = results_df.set_index("feature_set").loc["baseline"]
climate_metrics = results_df.set_index("feature_set").loc["baseline_plus_temperature"]

comparison = pd.DataFrame({
    "metric": ["MAE", "RMSE", "NMAE", "WAPE", "sMAPE"],
    "baseline": [baseline_metrics[m] for m in ["MAE", "RMSE", "NMAE", "WAPE", "sMAPE"]],
    "baseline_plus_temperature": [climate_metrics[m] for m in ["MAE", "RMSE", "NMAE", "WAPE", "sMAPE"]],
})

comparison["delta_climate_minus_baseline"] = (
    comparison["baseline_plus_temperature"] - comparison["baseline"]
)
comparison["relative_change_pct"] = (
    comparison["delta_climate_minus_baseline"] / comparison["baseline"] * 100
)

comparison


In [ ]:

top_imp = (
    importance_df.groupby(["feature_set", "feature"], as_index=False)["importance_mean"]
    .mean()
    .sort_values(["feature_set", "importance_mean"], ascending=[True, False])
)

display(top_imp.groupby("feature_set").head(15))


In [ ]:

for fs_name in importance_df["feature_set"].unique():
    plot_df = (
        importance_df[importance_df["feature_set"] == fs_name]
        .sort_values("importance_mean", ascending=False)
        .head(12)
        .sort_values("importance_mean")
    )

    plt.figure(figsize=(9, 5))
    plt.barh(plot_df["feature"], plot_df["importance_mean"])
    plt.title(f"Top permutation importances: {fs_name}")
    plt.xlabel("Importance mean")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()


In [ ]:

for fs_name in predictions_df["feature_set"].unique():
    plot_df = predictions_df[predictions_df["feature_set"] == fs_name].copy()

    plt.figure(figsize=(6, 6))
    plt.scatter(plot_df["y_true"], plot_df["y_pred"], alpha=0.6)
    max_val = max(plot_df["y_true"].max(), plot_df["y_pred"].max())
    plt.plot([0, max_val], [0, max_val], linestyle="--")
    plt.xlabel("Observed population")
    plt.ylabel("Predicted population")
    plt.title(f"Observed vs predicted: {fs_name}")
    plt.tight_layout()
    plt.show()



## 8. Interpretation checklist

Use this section after you run the notebook.

### If the climate model performs better
That suggests temperature is adding signal beyond short-term population history and metadata.

### If performance barely changes
That is still informative. It may mean:
- annual temperature at country level is too coarse
- the effect is species-specific rather than global
- climate acts with a longer lag than the current setup captures
- the current lag features already explain most short-term variation

### If climate features rank highly in permutation importance
That is evidence the model actually uses them, even if the aggregate metric improvement is modest.



## 9. Easy extensions

Because the notebook is modular, you can later add:
- precipitation
- drought indicators
- temperature anomaly instead of raw temperature
- rolling multi-year climate summaries
- climate extremes rather than annual means
- spatially finer climate data using latitude/longitude or subregion instead of country

The main pieces to extend are:
1. `load_<climate_variable>()`
2. `prepare_<climate_variable>()`
3. `merge_<climate_variable>()`
4. add new columns to `CLIMATE_NUMERIC_FEATURES`



## 10. Suggested next team discussion

A clean way to position this for the team:
- keep Steven's notebook as the **core benchmark**
- keep Lukas's notebook for **hyperparameter tuning**
- use this notebook as the **covariate integration / explainability track**

That way the work stays separated and reproducible.
